# 01 — Exploratory Data Analysis

This notebook performs exploratory analysis on the raw glycerol photocatalyst dataset.

In [ ]:
import os
import json
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

raw_dir = "../data/raw"
json_file = [f for f in os.listdir(raw_dir) if f.endswith('.json')][0]
fpath = os.path.join(raw_dir, json_file)

with open(fpath, 'r', encoding='utf-8-sig') as f:
    data = json.load(f)

df = pd.DataFrame(data)
print(f"Loaded {json_file}. Shape: {df.shape}")

In [ ]:
print("--- Dataset Shape ---")
print(df.shape)

print("\n--- Data Types ---")
print(df.dtypes.value_counts())

print("\n--- Missing Values % sorted descending ---")
null_pct = df.isna().mean() * 100
print(null_pct.sort_values(ascending=False).head(30))

In [ ]:
her_col = 'HER_std_umol_g_h'
plt.figure(figsize=(8, 5))
plt.hist(df[her_col].dropna(), bins=np.logspace(0, 6, 30), edgecolor='k', alpha=0.7)
plt.gca().set_xscale("log")
plt.xlabel("HER std (umol/g/h)")
plt.ylabel("Count")
plt.title("HER Distribution (Log X-axis)")
plt.show()

print("--- HER Percentiles ---")
percentiles = [0, 10, 25, 50, 75, 90, 99, 100]
for p in percentiles:
    val = np.percentile(df[her_col].dropna(), p)
    print(f"{p}th percentile: {val:.2f}")

In [ ]:
hash_col = 'experiment_hash'
if hash_col in df.columns:
    dup_groups = df.groupby(hash_col).filter(lambda x: len(x) > 1)
    print(f"Found {dup_groups[hash_col].nunique()} duplicate experiment_hash groups.")
    if len(dup_groups) > 0:
        display(dup_groups[[hash_col, 'full_catalyst', 'HER_std_umol_g_h', 'DOI']].head(15))
else:
    print("experiment_hash column not found.")

In [ ]:
if 'data_quality_flag' in df.columns:
    plt.figure(figsize=(6, 6))
    df['data_quality_flag'].value_counts().plot.pie(autopct='%1.1f%%', colors=['#4ade80', '#fbbf24', '#f87171'])
    plt.title("Data Quality Flag Breakdown")
    plt.ylabel("")
    plt.show()
else:
    print("data_quality_flag column not found.")

In [ ]:
df['log_HER'] = np.log1p(df['HER_std_umol_g_h'])
numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
# Exclude target helper
numeric_cols = [c for c in numeric_cols if c not in ['HER_std_umol_g_h', 'log_HER']]

corrs = df[numeric_cols].corrwith(df['log_HER'])
corrs_df = pd.DataFrame(corrs, columns=['corr_with_target']).sort_values(by='corr_with_target', ascending=False)

plt.figure(figsize=(5, max(4, len(corrs_df) * 0.3)))
sns.heatmap(corrs_df, annot=True, cmap='coolwarm', vmin=-1, vmax=1)
plt.title("Correlation with log1p(HER)")
plt.show()

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Count plot
df['host_material'].value_counts().head(10).plot.bar(ax=ax1, color='skyblue', edgecolor='k')
ax1.set_title("Top 10 Host Materials by Count")
ax1.set_ylabel("Count")
ax1.set_xlabel("Host")

# Median plot
df.groupby('host_material')['HER_std_umol_g_h'].median().loc[df['host_material'].value_counts().head(10).index].plot.bar(ax=ax2, color='salmon', edgecolor='k')
ax2.set_title("Top 10 Host Materials by Median HER")
ax2.set_ylabel("Median HER")
ax2.set_xlabel("Host")

plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(12, 6))
sns.heatmap(df.isna(), cbar=False, cmap='viridis')
plt.title("Missing Value Heatmap")
plt.show()

In [ ]:
if 'year' in df.columns:
    plt.figure(figsize=(10, 5))
    vc = df['year'].value_counts().sort_index()
    colors = ['red' if y == 1927.0 else 'skyblue' for y in vc.index]
    vc.plot.bar(color=colors, edgecolor='k')
    plt.xlabel("Year")
    plt.ylabel("Count")
    plt.title("Year Distribution (1927 highlighted red)")
    plt.show()
else:
    print("year column not found.")

In [ ]:
conf_cols = [c for c in df.columns if c.startswith('confidence_')]
if conf_cols:
    conf_data = {}
    for col in conf_cols:
        conf_data[col] = df[col].value_counts(normalize=True)
    
    conf_df = pd.DataFrame(conf_data).T
    # Ensure categories are ordered
    for cat in ['HIGH', 'MEDIUM', 'LOW']:
        if cat not in conf_df.columns:
            conf_df[cat] = 0.0
            
    conf_df = conf_df[['HIGH', 'MEDIUM', 'LOW']]
    conf_df.plot.bar(stacked=True, figsize=(10, 5), color=['#10b981', '#f59e0b', '#f43f5e'], edgecolor='k')
    plt.title("Confidence Scores Stacked Bar Chart")
    plt.ylabel("Proportion")
    plt.legend(title="Confidence Level")
    plt.show()
else:
    print("No confidence columns found.")